# Batch Scoring — Project Outcome Prediction

Scores every engagement; writes predictions + per-practice overrun risk summary.

**Reads:** `gold_ml_features` + saved model  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count, isnan, udf,
    sum as spark_sum, round as spark_round
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassificationModel

spark = SparkSession.builder.getOrCreate()
model = RandomForestClassificationModel.load('Files/models/overrun_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

numeric_features = [
    'budget_gbp', 'headcount', 'planned_duration_days', 'contract_value_gbp',
    'relationship_years', 'nps_score', 'lead_experience', 'lead_daily_rate',
]
cat_cols = ['practice', 'industry', 'tier', 'region', 'lead_grade']
indexed_df = df
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + [f'{c}_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df)

In [ ]:
scored = model.transform(model_df)
extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('overrun_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_overrun', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('overrun_probability') > 0.8, 'critical')
        .when(col('overrun_probability') > 0.6, 'high')
        .when(col('overrun_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('scored_at', current_timestamp())
    .select(
        'engagement_id', 'client_id', 'lead_consultant_id', 'practice', 'industry', 'tier',
        'budget_gbp', 'headcount', 'planned_duration_days',
        'had_overrun', 'predicted_overrun', 'overrun_probability', 'risk_level',
        'scored_at')
)
predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-practice overrun risk summary
summary = (
    predictions
    .groupBy('practice')
    .agg(
        count('*').alias('total_engagements'),
        spark_sum('predicted_overrun').alias('predicted_overrun_count'),
        spark_sum('had_overrun').alias('actual_overrun_count'),
        spark_round(avg('overrun_probability'), 4).alias('avg_overrun_probability'),
        spark_round(avg('budget_gbp'), 2).alias('avg_budget_gbp'),
        spark_round(avg('planned_duration_days'), 1).alias('avg_duration_days'),
    )
    .withColumn('overrun_rate', spark_round(col('predicted_overrun_count') / col('total_engagements') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_overrun_probability') > 0.6, 'high')
        .when(col('avg_overrun_probability') > 0.3, 'medium')
        .otherwise('low'))
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_overrun_probability').desc())
)
summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Practice overrun summary: {summary.count()} rows')
summary.show(15, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('All Gold ML tables optimized')